# clinker substitution cement retrofit NPV simulation

Run the clinker substitution cement retrofit Monte Carlo simulation and visualize the resulting NPV distribution.

The summary also reports how many simulations have non-negative NPV (NPV >= 0) and how many have negative NPV.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cement.cement_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_RETROFIT_BAU_MODE,
    DEFAULT_SAMPLE_SIZE,
    simulate_cement_results,
)

from npv_summary import summarize_metric_signs


In [2]:
TECHNOLOGY = 'clinker_substitution'
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED
RETROFIT_BAU_MODE = DEFAULT_RETROFIT_BAU_MODE

results_by_technology = simulate_cement_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=(TECHNOLOGY,),
    retrofit_bau_mode=RETROFIT_BAU_MODE,
)
simulation = results_by_technology[TECHNOLOGY]
results = pd.DataFrame(simulation)
results.head()


,run_id,technology,technology_type,retrofit_bau_mode,annual_output_t,lifetime_years,capex_eur_per_t,fixed_opex_eur_per_t,variable_opex_eur_per_t,fuel_consumption_mwh_th_per_t,...,bau_variable_opex_eur_per_t,bau_fuel_consumption_mwh_th_per_t,bau_electricity_consumption_mwh_per_t,bau_emissions_tco2_per_t,capex_change_eur_per_t,fixed_opex_change_eur_per_t,variable_opex_change_eur_per_t,fuel_consumption_reduction_fraction,electricity_consumption_reduction_fraction,emissions_reduction_fraction
0,0,clinker_substitution,retrofit,sampled,1000000.0,25.0,165.479121,14.937909,8.535307,0.552598,...,5.135369,0.657332,0.084504,0.620035,0.0,0.0,3.399937,0.159333,0.0,0.095299
1,1,clinker_substitution,retrofit,sampled,1000000.0,25.0,158.777569,14.946918,8.657300,0.491563,...,5.125930,0.649577,0.093378,0.602551,0.0,0.0,3.531370,0.243257,0.0,0.111943
2,2,clinker_substitution,retrofit,sampled,1000000.0,25.0,167.171958,14.464250,8.525759,0.535983,...,5.431670,0.697878,0.085678,0.683904,0.0,0.0,3.094089,0.231982,0.0,0.094682
3,3,clinker_substitution,retrofit,sampled,1000000.0,25.0,163.947361,14.001728,10.947041,0.566254,...,5.260010,0.693428,0.089539,0.634547,0.0,0.0,5.687031,0.183399,0.0,0.187291
4,4,clinker_substitution,retrofit,sampled,1000000.0,25.0,151.883547,14.719678,9.868968,0.534034,...,5.371749,0.694798,0.084480,0.652753,0.0,0.0,4.497220,0.231383,0.0,0.098530


In [3]:
npv_million_eur = results["npv_eur"] / 1_000_000
lcoc_eur_per_t = results["lcoc_eur_per_t"]
levelized_net_margin_eur_per_t = results["levelized_net_margin_eur_per_t"]

summary = pd.concat(
    [
        npv_million_eur.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "NPV million EUR"
        ),
        lcoc_eur_per_t.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "LCOC EUR/t cement"
        ),
        levelized_net_margin_eur_per_t.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "Levelized net margin EUR/t cement"
        ),
    ],
    axis=1,
)

npv_signs = summarize_metric_signs(npv_million_eur)
npv_sign_summary = pd.DataFrame(
    {
        "NPV category": ["Non-negative (NPV >= 0)", "Negative (NPV < 0)"],
        "Simulation count": [
            npv_signs["non_negative_count"],
            npv_signs["negative_count"],
        ],
        "Simulation share": [
            npv_signs["non_negative_share"],
            1.0 - npv_signs["non_negative_share"],
        ],
    }
)

display(summary)
display(npv_sign_summary)


,NPV million EUR,LCOC EUR/t cement,Levelized net margin EUR/t cement
count,100000.000000,100000.000000,100000.000000
mean,469.704638,105.998643,44.001357
std,49.253364,4.613995,4.613995
min,288.644086,89.644673,27.039826
5%,388.655424,98.355568,36.408766
50%,469.636621,106.005015,43.994985
95%,551.292756,113.591234,51.644432
max,644.279612,122.960174,60.355327


,NPV category,Simulation count,Simulation share
0,Non-negative (NPV >= 0),100000,1.0
1,Negative (NPV < 0),0,0.0


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(npv_million_eur, bins=50, color="tab:gray", edgecolor="white", alpha=0.8)
ax.axvline(npv_million_eur.mean(), color="tab:blue", linewidth=2, label="Mean")
ax.axvline(npv_million_eur.median(), color="tab:orange", linewidth=2, label="Median")
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"clinker substitution cement retrofit NPV distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("NPV (million EUR)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42708/170796050.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    levelized_net_margin_eur_per_t,
    bins=50,
    color="tab:green",
    edgecolor="white",
    alpha=0.8,
)
ax.axvline(
    levelized_net_margin_eur_per_t.mean(),
    color="tab:blue",
    linewidth=2,
    label="Mean",
)
ax.axvline(
    levelized_net_margin_eur_per_t.median(),
    color="tab:orange",
    linewidth=2,
    label="Median",
)
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"clinker substitution cement retrofit levelized net margin distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("Levelized net margin (EUR/t cement)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42708/3369878837.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
annual_components = results[
    [
        "annual_revenue_eur",
        "annual_fixed_opex_eur",
        "annual_variable_opex_eur",
        "annual_fuel_cost_eur",
        "annual_electricity_cost_eur",
        "annual_emissions_cost_eur",
        "annual_net_cash_flow_eur",
    ]
] / 1_000_000

annual_components.mean().rename("Mean annual value, million EUR")


annual_revenue_eur             150.000000
annual_fixed_opex_eur           14.332408
annual_variable_opex_eur         9.945767
annual_fuel_cost_eur             6.460156
annual_electricity_cost_eur     15.926099
annual_emissions_cost_eur       44.344437
annual_net_cash_flow_eur        58.991133
Name: Mean annual value, million EUR, dtype: float64

In [7]:
retrofit_columns = [
    "capex_change_eur_per_t",
    "fixed_opex_change_eur_per_t",
    "variable_opex_change_eur_per_t",
    "fuel_consumption_reduction_fraction",
    "electricity_consumption_reduction_fraction",
    "emissions_reduction_fraction",
    "bau_capex_eur_per_t",
    "bau_fixed_opex_eur_per_t",
    "bau_variable_opex_eur_per_t",
    "bau_fuel_consumption_mwh_th_per_t",
    "bau_electricity_consumption_mwh_per_t",
    "bau_emissions_tco2_per_t",
]

available_retrofit_columns = [column for column in retrofit_columns if column in results]
retrofit_summary = results[available_retrofit_columns].describe(
    percentiles=[0.05, 0.5, 0.95]
)
retrofit_summary


,capex_change_eur_per_t,fixed_opex_change_eur_per_t,variable_opex_change_eur_per_t,fuel_consumption_reduction_fraction,electricity_consumption_reduction_fraction,emissions_reduction_fraction,bau_capex_eur_per_t,bau_fixed_opex_eur_per_t,bau_variable_opex_eur_per_t,bau_fuel_consumption_mwh_th_per_t,bau_electricity_consumption_mwh_per_t,bau_emissions_tco2_per_t
count,100000.0,100000.0,100000.000000,100000.000000,100000.0,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.0,0.0,4.778544,0.199973,0.0,0.124846,160.012499,14.332408,5.167223,0.666500,0.086635,0.633377
std,0.0,0.0,1.028074,0.028924,0.0,0.043257,5.770418,0.471200,0.235343,0.040049,0.004713,0.023522
min,0.0,0.0,3.000014,0.150003,0.0,0.050001,150.000263,13.009529,4.504187,0.610000,0.080000,0.600000
5%,0.0,0.0,3.178734,0.154934,0.0,0.057545,150.989862,13.451564,4.725440,0.614265,0.080499,0.602579
50%,0.0,0.0,4.774618,0.199872,0.0,0.124744,160.049191,14.411204,5.207742,0.659535,0.085806,0.629393
95%,0.0,0.0,6.382570,0.245005,0.0,0.192292,169.007981,14.949464,5.474133,0.741932,0.095517,0.677623
max,0.0,0.0,6.559994,0.250000,0.0,0.199999,169.999875,14.999998,5.499993,0.779352,0.099946,0.699866
